### 规格化欧盟动物数据

In [1]:
import pandas as pd
# Load the new CSV file 
for y in range(1975, 2024):
    file_path = 'D:/中科院数据下载/eurostat/crops/'+str(y)+'.csv'

    crops = pd.read_csv(file_path)

    # 去除一些多余的行
    items = [] # 需要删除的行
    data_with_specific_item = crops[crops["strucpro_label"].str.contains("%|tonne/ha")]
    items.extend(list(data_with_specific_item["strucpro_label"].unique()))


    crops = crops.drop(crops[crops['strucpro_label'].isin(items)].index)
    crops["year"] = y
    new_order = ['geo\TIME_PERIOD', 'geo_label','year', 'crops', 'crops_label', 'strucpro', 'strucpro_label', str(y)]
    crops = crops.reindex(columns=new_order)

    # Check the unique values columns to identify the indicators for harvested area, yield, and planted area
    unique_categories = crops['strucpro_label'].unique()

    # Creating a new column to distinguish between different data types (Harvested Area, Yield, Planted Area, etc.)
    crops['DataType'] = crops['strucpro_label'] + '_' + crops['crops_label']

    # 行政区格式化
    # 地区标准化排列
    # for y in range(1977,2024):
    # crops = pd.read_csv("animals_差禽类/"+str(y)+".csv")
    # 提取国家、州、县形成字典
    geo_dict = crops.set_index("geo\TIME_PERIOD")["geo_label"].to_dict()

    # 提取县
    # crops_county = crops[crops["geo\TIME_PERIOD"].str.contains("\d{2}$",regex=True)]
    crops_county = crops[crops["geo\TIME_PERIOD"].str.len() == 4]
    crops_county = crops_county.rename(columns={"geo\TIME_PERIOD":"county","geo_label":"county_label"})

    # 补全县前面的州
    crops_county.loc[:, 'state'] = crops_county['county'].str[:-1]
    crops_county.loc[:, 'state_label'] = crops_county['state'].map(geo_dict)

    # 将新列移动到最左边
    cols = crops_county.columns.tolist()
    cols.insert(0, cols.pop(cols.index('state_label')))
    cols.insert(0, cols.pop(cols.index('state')))
    crops_county = crops_county.reindex(columns=cols)

    # 补全县前面的国家
    crops_county.loc[:, 'country'] = crops_county['state'].str[:-1]
    crops_county.loc[:, 'country_label'] = crops_county['country'].map(geo_dict)

    # 将新列移动到最左边
    cols = crops_county.columns.tolist()
    cols.insert(0, cols.pop(cols.index('country_label')))
    cols.insert(0, cols.pop(cols.index('country')))
    crops_county = crops_county.reindex(columns=cols)

    # 提取州
    # crops_state = crops[crops["geo\TIME_PERIOD"].str.contains("^[^\d]*\d{1}$", regex=True)]
    crops_state = crops[crops["geo\TIME_PERIOD"].str.len() == 3]

    crops_state = crops_state.rename(columns={"geo\TIME_PERIOD":"state","geo_label":"state_label"})

    # 补全州前面的国家
    crops_state.loc[:, 'country'] = crops_state['state'].str[:-1]
    crops_state.loc[:, 'country_label'] = crops_state['country'].map(geo_dict)

    # 将新列移动到最左边
    cols = crops_state.columns.tolist()
    cols.insert(0, cols.pop(cols.index('country_label')))
    cols.insert(0, cols.pop(cols.index('country')))
    crops_state = crops_state.reindex(columns=cols)

    # 用空值补全县的列
    crops_state.insert(4,"county","")
    crops_state.insert(5,"county_label","")

    # 提取国家
    # crops_country = crops[crops["geo\TIME_PERIOD"].str.contains(".*\D$", regex=True)]
    crops_country = crops[crops["geo\TIME_PERIOD"].str.len() == 2]
    
    crops_country = crops_country.rename(columns={"geo\TIME_PERIOD":"country","geo_label":"country_label"})

    crops_country.insert(2,"state","")
    crops_country.insert(3,"state_label","")
    crops_country.insert(4,"county","")
    crops_country.insert(5,"county_label","")

    # 合并
    crops = pd.concat([crops_county,crops_state,crops_country],axis=0,ignore_index=True)

    # Pivot the table
    pivot_data_2015 = crops.pivot_table(
        index=['country',  'country_label', 'state', 'state_label','county','county_label','year'], 
        columns='DataType', 
        values=str(y), 
        aggfunc='sum'
    )
        
    # Reset the index so that year, State, County, and OBJECTID are columns again
    pivot_data_2015_reset = pivot_data_2015.reset_index()

    # File path for the new pivoted CSV file (2015)
    pivoted_file_path_2015 = '农作物_ok/'+str(y)+'.csv'

    # Export the pivoted data to a CSV file
    pivot_data_2015_reset.to_csv(pivoted_file_path_2015, encoding='utf-8-sig', index=False)


In [ ]:
import pandas as pd
# Load the new CSV file 
for y in range(1909, 2024):
    file_path = 'standard/county/农产品/'+str(y)+'.csv'
    crops = pd.read_csv(file_path)

    # 去除一些多余的行
    items = [] # 需要删除的行
    data_with_specific_item = crops[crops["Data Item"].str.contains("IRRIGATED")]
    items.extend(list(data_with_specific_item["Data Item"].unique()))
    
    crops = crops.drop(crops[crops['Data Item'].isin(items)].index)

    # Check the unique values in 'Category' and 'Data Item' columns to identify the indicators for harvested area, yield, and planted area
    unique_categories = crops['Category'].unique()
    # unique_data_items = data_2015['Data Item'].unique()

    # Convert the 'Value' column to numeric after removing commas
    crops['Value'] = pd.to_numeric(crops['Value'].str.replace(',', ''), errors='coerce')

    # Creating a new column to distinguish between different data types (Harvested Area, Yield, Planted Area, etc.)
    crops['DataType'] = crops['Category'] + '_' + crops['Commodity']

    # Pivot the table
    pivot_data_2015 = crops.pivot_table(
        index=['year',  'OBJECTID', 'State', 'County'], 
        columns='DataType', 
        values='Value', 
        aggfunc='sum'
    )

    # Reset the index so that year, State, County, and OBJECTID are columns again
    pivot_data_2015_reset = pivot_data_2015.reset_index()

    # File path for the new pivoted CSV file (2015)
    pivoted_file_path_2015 = '农作物_ok/'+str(y)+'.csv'

    # Export the pivoted data to a CSV file
    pivot_data_2015_reset.to_csv(pivoted_file_path_2015, encoding='utf-8-sig', index=False)



In [12]:
import pandas as pd
data = pd.read_csv("D:/中科院数据下载/eurostat/crops/1975.csv")
data = data.loc[4000:4010]

In [16]:
data.dropna(how='all', inplace=True)
data.dropna(axis=1)                                                                                                                                           

,freq,crops,crops_label,strucpro,strucpro_label,geo\TIME_PERIOD,geo_label
4000,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR52,Bretagne (NUTS 2013)
4001,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR53,Poitou-Charentes (NUTS 2013)
4002,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR6,Sud-Ouest (FR) (NUTS 2013)
4003,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR61,Aquitaine (NUTS 2013)
4004,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR62,Midi-Pyrénées (NUTS 2013)
4005,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR63,Limousin (NUTS 2013)
4006,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR7,Centre-Est (FR) (NUTS 2013)
4007,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR71,Rhône-Alpes (NUTS 2013)
4008,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR72,Auvergne (NUTS 2013)
4009,A,C1110,Common wheat and spelt,AR,Area (cultivation/harvested/production) (1000 ha),FR8,Méditerranée (NUTS 2013)


In [19]:
from sklearn.impute import SimpleImputer
import numpy as np 
im = SimpleImputer(missing_values=np.nan, strategy='mean')
data = np.array([
    [1,2,3],
    [np.nan,5,6],
    [7,np.nan,9]
])
im.fit_transform(data)


array([[1. , 2. , 3. ],
       [4. , 5. , 6. ],
       [7. , 3.5, 9. ]])